# Глава 6. Дообучение

В предыдущей главе мы рассмотрели первый и самый важный этап обучения больших языковых моделей - предобучние, а также сделали обзор подходов к подготовке обучающих датасетов.

В данной главе рассмотрим второй этап обучения языковых моделей: дообучение с учителем (Supervised Fine-tuning). Перечислим основные узкие места и методы борьбы со сложностью дообучения, а также дадим подробный обзор подходов эффективного дообучения PEFT (parameter efficient fine-tuning).

### Введение
Pretrain обучение запускалось в режиме самообучения (self-supervised) на некотором универсальном датасете текстов. Полученную в результате модель будем назвывать предобученной (pretrained). Такая модель хорошо умеет предсказывать следующий токен в предложении, обладает богатыми «общими знаниями» о языке и мире, но совершенно не пригодна для выполнения узких специфических задач, где важна структура ответа: она не умеет следовать инструкциям, не знает доменную специфику, не умеет форматировать ответы в требуемом формате и тоне. Такую модель можно использовать для базовых задач, но ее польза сильно ограничена

Для того чтобы обойти это ограничение, модель почти всегда дополнительно обучают под конкретную задачу. Благо, для дообучения нужно в разы меньше данных, чем для обучения с нуля. Для примера, модель GPT-3 в 2020 году предобучалась на датасете в 300 млрд токенов, а для получения ее инструктивной версии InstructGPT потребовалось дообучение на всего лишь 10 млн токенов (13 тысяч примеров). Иными словами, доводка модели требует примерно 0.01% от объема данных для pretrain. А если дообучать через промптинг, то и вообще можно ограничиться несколькими десятками токенов. Проводя аналогию с обучением человека: для ребенка, который научился читать и писать, и овладел базовыми навыками, изучение нового предмета становится кратно более легкой задачей

Есть несколько режимов адаптации предобученной модели под задачу - "временный", через прописывание инструкций в промпт и "постоянный", через обновление весов модели. Опишем их:
- Zero-shot learning<br>в этом режиме мы никак не модифицируем модель, надеемся что она "из коробки" сумеет сгенерировать устраивающий нас ответ; если задача не сложная или качество ответа не критичный показатель, это бывает вполне оправданно<br><br>
- Few-shot learning<br>мы добавляем в промпт модели один или несколько примеров в том формате, который от нее ожидаем ("сделай вот как здесь", "отоформатируй ответ вот так и так"). Интеллекта модели хватает, чтобы неплохо обобщить требования из нескольких примеров<br><br>
- Supervised Fine-tuning<br>мы проводим дополнительный цикл обучения модели на относительно небольшом размеченном наборе данных (1-100K примеров). Идея в том, что модель уже обладает фундаментальными языковыми навыками, и нам остаётся лишь донастроить её под задачу

Дообучать модель можно как под одну задачу, так и сразу под несколько __Multi-task Fine-tuning__.  В первом случае получается узкоспециализированная модель, во втором универсальная. Одной из первых попыток построить подобную универсальную модель была модель __T5__ из Google [(Raffel et al, 2019)](https://arxiv.org/abs/1910.10683), которую обучали сразу на 62 размеченных датасетах. Большинство современных моделей также пошли по второму пути, для этого датасет для дообучения стараются компоновать из набора разнородных задач.

### Проблемы дооубчения
Несмотря на меньшие требования по кол-ву примеров по сравнению с предобучением в self-supervised режиме, это все же обучение с учителем, что делает процесс крайне дорогим:
- ручная разметка - это всегда дорого<br>нужно платить разметчикам; чем сложнее задача (например, математика, физика), тем дороже стоит время эксперта; размечают обычно с перекрытием, чтобы обеспечить качество данных; нужен контроль, чтобы избежать накруток; нужно грамотно составить пул задач, чтобы он был достаточно реперезентативен<br><br>
- дорого по памяти<br>каждый цикл обучения модели требует хранить все ее веса, градиенты, состояния оптимизатора. Даже для небольших моделей это 10-100GB видеопамяти<br><br>
- дорого по занимаемому месту на диске<br>каждая дообученная версия — это полная копия исходной модели. Все версии нужно где-то хранить, даже если там поменялась пара весов<br><br>
- есть проблема забывания (Catastrophic forgetting)<br>при долгом обучении модель может «забывать» часть исходных способностей. Мы же хотим чтобы модель "накапливала" знаяния, а не переписывала их

Феномен *Catastrophic forgetting* был впервые описан [(MacCloskey et al, 1989)](https://www.andywills.info/hbab/mccloskeycohen.pdf) на примере нейронной сети, обучавшейся сложению двух чисел - когда после обучения складывать однозначные числа модели обучали сложению двузначных, они полностью разучивались складывать однозначные. Иными словами, посмотрел рилс - забыл дату куликовской битвы. В реальности конечно чуть сложнее, но тем не менее - обучение, тем более если оно многошаговое, гетерогенное (учат разным навыкам) и растянутое по времени, нужно грамотно планировать.

В контексте обучения языковых моделей проблема забывания еще больше актуальна для этапа Preference Optimization (в следующей главе), там ограничение на величину обновления весов прямо вшито на уровне алгоритма. Авторы InstructGPT называли этот феномен *alignment tax* - чем больше мы подстраиванием модель под пользователя, тем больше это происходит за счет некоторой потери качества

Полезность этапа дообучения с учителем многократно ислледовалась. Например, в работе [(Shi et al., 2023)](https://arxiv.org/abs/2308.04014) посчитали влияние дообучения на качество модели LLaMA‑7B. На целевом датасете оно ожидаемо выросло, а вот общие навыки по бенчмарку MMLU упали на –3.5 процентных пункта

Некоторые методы борьбы с забыванием:
- *Replay / Mixing*<Br>При дообучении на новом датасете периодически подмешивают данные из pre‑training датасета<br><Br>
- *Регуляризация*<br>вариант, предложенный [(Kirkpatrick et al, 2017)](https://arxiv.org/pdf/1612.00796) который назвали Elastic Weight Consolidation (EWC) - в функцию потерь добавляется штраф за большое изменение параметра (было vs стало), оцененный по метрике Fisher Information; похожая регуляризация использовалась и в первых моделях пользовательских предпочтений типа RLHF (подробнее в следующей главе)<br><br>
- *LoRA и адаптеры*<Br>обучают только небольшие низкоранговые матрицы, не трогая основные веса. Значительно снижает забывание, но при агрессивном обучении всё же возможно; подробнее о методах репараметризации ниже<br><Br>
- *Мультзадачное обучение* (multi-task learning)<Br>если дообучать модель нужно сразу под несколько задач, лучше делать это не последовательно, а чередовать примеры из каждого датасета, так модель улавливает больше полезного сигнала<br><br>
- *Постепенное размораживание*<br>начинать с последних слоёв, оставляя ранние нетронутыми. Полезно при адаптации к новым доменам; снижает падение метрик качества относительно полного дообучения

Многочисленные ислледования, например, [(Frankie et al, 2019)](https://arxiv.org/abs/1803.03635) показывают, что современные нейросетевые модели избыточны в точки зрения хранения информации в своих весах. Иными словами, большую часть весов можно удалить или занулить и качество модели почти не просядет. На этой идее базируется целое направление - прунинг Pruning нейросестей, который выолняется на этапе постпроцессинга после обучения. Его идея в том, чтобы после обучения эмпирически выделять наиболее бесполезные параметры и убирать их из модели, делая саму модель кратно меньше по объему. К сожалению текущие подходы не позволяют узнать до начала обучения, какой параметр будет полезен, а какой нет. Поэтому прунинг не решает проблему полностью, да и для практического использования подход неудобен, так как требует железа, умеющего эффективно работать с разреженностью.

Итого, дообучение модели - дорогое, но его точно можно сделать более дешевым. Вопрос - как?

## Методы эффективного дообучения

PEFT (Parameter-Efficient Fine-Tuning) — это зонтичный термин для семейства подходов, объединённых общей идеей: заморозить большую часть весов предобученной модели и обучать лишь маленькую долю параметров (часто менее 1%). Суть в том, что предобученная модель уже содержит почти всё нужное и чтобы адаптировать её под задачу, не нужно обновлять всю сеть целиком — достаточно небольшого «адаптера», который скорректирует прохождение сигнала ближе к целевому поведению модели

Преимущества PEFT-подходов:
- так как большинство весов заморожено, обучается лишь небольшое кол-во параметров, а это значит, что на обучении нужно хранить мало градиентов и состояний оптимизатора => резко снижается потребление памяти
- дообученный «адаптер» весит мегабайты вместо гигабайт, а значит его легко хранить и подключать при необходимости к основной модели (аналог "картриджа", вставляемого в устройство)
- веса оригинальной модели не изменяются, а значит меньше риск переобучиться на подаваемой обучающей выборке и меньше забывания => модель работает точнее

Благодаря своей простоте идея породила целую область исследований, в рамках которой появилось множество различных подходов, отличающихся тем, что именно замораживаем, что именно обучаем, как агрегируем. В 2024 году вышло исследование [(Lialin et al, 2024)](https://arxiv.org/pdf/2303.15647) с обзором существующих на тот момент PEFT методов. Многие из этих подходов будут обзорно рассмотрены ниже. Практика показала, что наибольшую популярность приобрели методы семейства LoRA (low rank adaptation), о которых речь пойдет ниже в этой главе.

### Таксономия PEFT-методов

Все PEFT-методы можно разложить по нескольким классам в зависимости от того, *что именно* в модели становится обучаемым. «добавляем новые параметры или выбираем существующие?», «если добавляем — модулем внутри сети или вектором на входе?», «меняем веса напрямую или через компактную репараметризацию?»

<img src="img/finetuning_peft.png" width=600>

Обычно выделяют следующие группы подходов:
- *селективные*: <br>большую часть весов модели замораживаем, выбираем небольшой процент параметров, которые считаем значимыми и дообучаем модель, обновляя только их
- *аддитивные*: веса модели замораживаем, добавляем новые параметры, цель которых корректировать сигнал под задачу, и дообучаем модель, обновляя только их
- *адаптерные*: веса модели замораживаем, добавляем новые параметры, в виде "адаптера" - небольшой двухслойной MLP сети, которая кодируют апдейт
- *промптинговые (Soft prompts)*: веса модели замораживаем, присоединяем в модель фиксированное кол-во эмбедингов, отвечающих за выбор режима работы модели
- *репараметризующие*: веса модели замораживаем, добавляем адаптер, мультипликативное приближение
- *Гибридные*: комбинация нескольких подходов

## Селективные методы

### Введение
Это группа реализует, пожалуй, самую простую идею: давайте заморозим большую часть весов, а дообучать будем небольшое подмножество ее параметров, которые мы по какому-то критерию опредлеим как наиболее значимые. Выбор критерия значимости остается за разработчиком метода, некоторые критерии можно посчитать на препроцессинге, перед обучением, другие только на пост-процессинге, когда выполнили полный цикл обучения. Выбор параметров обычно кодируется бинарной маской [0,1], которая показывает, "выбран" или "не выбран" параметр для обновления.

К сильным сторонам этой группы можно отнести то, что исходная архитектура модели при этом никак не меняется, нет дополнительных слоёв, которые нужно вычислять.

А к слабым сторонам то, что во-первых точечная заморозка отдельных весов крайне неэффективна с точки зрения реализации на видеокартах, которые просто не предназначены для этого, на видеокартах все заточено именно под блочные вычисления. Во-вторых, методы, основанные на постпроцессинге, требуют отдельного цикла обучения, что может быть даже дороже полного обучения.

Важно отметить, что в ранних PEFT методах, к которым безусловно относятся селективные, основной ценностью была именно возможность комбинировать знания, то есть подключать по необхоимости новые "навыки" через прибавку дельты весов. Но с ростом моделей на первый план вышла пиковая память при обучении, и селективные методы проиграли другим группам именно по этому параметру.

### Обзор методов

Начнем с выбора параметров по их функциональному назнчению, это самая простая логика. [(Zaken et al, 2022)](https://arxiv.org/abs/2106.10199) в **BitFit** обучают только "свободные" параметры смещения (biases). Аналогично можно обучать только параметры слоев нормализации как в **LN Tuning** [(Zhao et al, 2020)](https://arxiv.org/abs/2312.11420) или только веса слоёв внимания как в **Attention Tuning**.

Вторая полгруппа методов реализует отбор параметров для обновления на основании расчетных критерив. Тут много пересесчений с теорией прунинга нейросетей, поскольку и там, и тут решается очень похожая задача. Там цель - занулить как можно больше параметров модели, тут заморозить для обновления. Следующие несколько методов основаны именно на этой идее.



Метод **LT-SFT** [(Ansell et al., 2022)](https://arxiv.org/abs/2110.07560) черпает вдохновение из теории прунинга нейросетей, а конкернто из упомянутой в начале главы концепции Lottery Ticket Hypothesis, описанной (Frankle & Carbin, 2019), которая утверждает, что почти все реальные модели избыточны с точки зрения кол-ва параметров и для эффективного обучения достаточно их небольшого процента.

Также как и в оригинальной работе, предложенный авторами метод работает в две фазы. Сначала модель полностью дообучают на некоторой узкой задаче, получая значения параметров $\theta^{(1)}$. Затем модель оценивают и отбирают топ $k$ параметров больше всего изменившихся с начала обучения $|\theta^{(1)}_i - \theta^{(0)}_i|$, их считаем наиболее значимыми для модели. На втором шаге все веса откатывают обратно к $\theta^{(0)}$ и модель дообучают повторно, но уже обновляют только k отобранных параметров. Полученная в итоге разность $\delta = \theta^{(2)} - \theta^{(0)}$ и есть итоговый разреженный вектор апдейта.

Откат к исходным весам гарантирует в том числе, что все дельты отсчитываются от одной базы, то есть живут в общем пространстве параметров. Благодаря этому композиция нескольких дообучений сводится к обычному сложению векторов, и модель остаётся аддитивной по добавляемым знаниям. Авторы демонстрируют этот эффект на примере zero-shot переноса между языками: первый вектор апдейта обучают на неразмеченном тексте нужного языка (задача восстановления замаскированных токенов), второй — на размеченных данных задачи, доступных только на английском. Складывая эти два "навыка" мы получаем новую модель $\theta^{(0)} + \delta_{lang} + \delta_{task}$, способную решать задачу на языке, для которого разметки не было вовсе. И хотя аддитивность довольно стандартный эффект, подчеркивается, что она особенно хорошо работает в условиях разреженности, так как наложение параметров минимально и апдейты не мешают друг другу.

В методе **Fish Mask** [(Sung et al., 2021)](https://arxiv.org/abs/2111.09839) маску выбирают один раз *до* начала дообучения и больше не трогают. Важность параметра оценивают таким показателем из статистики, как информация Фишера: $$\hat F_i = \frac{1}{N}\sum_{n=1}^{N}\left(\frac{\partial \log p(y_n \mid x_n; \theta)}{\partial \theta_i}\right)^2$$

Для подсчета достаточно знать градиент предобученной модели и для его вычисления модель прогоняют на 100 примерах. На втором шаге берут $k$ параметров с наибольшим $\hat F_i$, фиксируют маску и дообучают только их, остальные веса заморожены. Это сознательное упрощение для ускорения расчета, здесь мы ориентируемся не на накопленный сигнал, а на локальный ландшафт в стартовой точке.

Интерпретация такая: мы попадаем в какую-то стартовую точку на поверхности правдоподобия (применительно к узкой задаче, это не обязательно оптимум) и осматриваемся. Выбираем k осей, вдоль которых функция правдоподобия (она же функция потерь) меняется наиболее радикально и оставляем только эти оси, значения по остальным замораживаем. Далее запускаем алгоритм оптимизации. Название Fish Mask расшифровывается как Fisher-Induced Sparse uncHanging mask.

Метод **FAR** (Freeze And Reconfigure) от [(Vucetic et al., 2022)](https://arxiv.org/abs/2205.01541) реализует идею похожую на LT-SFT, но делает акцент на неструктурированности подмножества параметров. В дообучении на edge-устройствах, узкое место - это не размер сохраняемого чекпоинта, а объем используемой памяти и время вычислений. Неструктурированная разреженность (рандомные параметры) невыгодна, потому что разбросанные по матрице индексы дают нерегулярные обращения к памяти. Поэтому FAR отбирает не отдельные веса, а целые узлы (строки) линейных слоёв в FFN-блоках. Как мы отмечали в главе про Трансформеры, на FFN приходится больше 2/3 параметров модели.

Сначала для вычисления статистик делается короткий пробный прогон обучения, который авторы называют *priming*, и который состоит из порядка 1% от общего числа шагов. Затем для каждого нейрона считают показатель $m^{e,i}_n = \lVert \phi^{e,i}_n - w^{e,i}_n \rVert_1$ — насколько сильно он сдвинулся с начала обучения. Топ $r$ узлов объявляют обучаемыми, остальные замораживают. Дальше происходит шаг "реконфигурации": замороженные и незамороженные узлы разносят в два параллельных подмодуля, и мы можем воспользоваться блочностью вычислений на видеокарт и усорить расчет. Оригинальный порядок восстанавливают перестановкой.

В методе **Diff-Pruning** [(Guo et al., 2021)](https://arxiv.org/abs/2012.07463) авторы вдохновились классическим алгоритмом прунинга от (Louizos et al., 2018) и адаптировали его под задачу supervised fine-tuning. Выпишем, как происходит изменение весов модели через вектор апдейта: $\theta_{task} = \theta_{pretrain} + \delta.$ Чтобы хранить этот вектор дёшево, потребуем занулить в нём как можно больше компонент, представив его как $\delta = z \odot w$, где $z$ - бинарная маска $z \in [0,1]$. Разреженность вектора измеряется $L_0$-нормой (числом ненулевых компонент), которая по определению недифференцируема, поэтому авторы заменяют маску $z$ непрерывным приближением: в ней хранятся не дискретные метки 0/1, а какое-то число из интервала [0,1]. Матожидание $L_0$-нормы этого приближения добавляют штрафом в функцию потерь, которую уже можно оптимизировать стандартными методами.

Приближение рассчитывается как сигмоида от суммы обучаемого параметра $\alpha$ и случайного логистического шума, растянутая за пределы $[0,1]$ и обрезанная порогом. Растяжение делается, чтобы дать ненулевую вероятность граничных значений, нуля и единицы. Приятное свойство, что при такой формулировке матожидание $L_0$-нормы удобно записывается формулой как функция параметра $\alpha$:

$$\mathbb{E}_{u}\big[\|\delta\|_0\big] = \sum_{i=1}^{d} P(z_i \neq 0) = \sum_{i=1}^{d} \sigma\!\left(\alpha_i - \tau \log \frac{-l}{r}\right)$$

Поэтому весь функционал (функция потерь + штраф) теперь может оптимизироваться обычным градиентным спуском. После обучения шум убирают, маску замораживают и найденный оптимум проецируют в ближайшую точку с заданным количеством ненулевых параметров. После этого ненулевые параметры ещё раз обучают и получившийся вектор - это и есть итоговый разреженный вектор апдейта $\delta$.


## Аддитивные методы

### Введение
И адаптеры, и софт-промпты добавляют обучаемые параметры внутрь вычислительного графа основной сети. У них есть общее ограничение: раз обучаемый модуль находится внутри модели, при обучении на обратном проходе градиент обязан пройти через всю сеть до самого нижнего слоя. А это значит, что нужно хранить все промежуточные активации от всех слоёв. И хотя число обучаемых параметров падает на два порядка, использование памяти сокращается лишь процентов на тридцать. 

Методы этого подраздела также добавляют новые параметры, но размещают их вне графа обратного распространения основной сети. Причем все рассматриваемые далее методы агрегируют сигнал не только от последнего, а от всех слоев (кто-то агрегирует рекурсивно, кто-то в attention стиле, кто-то через систему регулирования). Идея в том, что разные слои трансформера кодируют признаки разной степени абстракции, и разным задачам нужны разные слои.

### Обзор методов

#### Ladder-Side Tuning

Метод **Ladder-Side Tuning** от [(Sung et al., 2022)](https://arxiv.org/abs/2206.06522) как бы достраивает отдельную небольшую «боковую» сеть, которая обучается параллельно замороженной модели, получая на вход её скрытые состояния.  

На каждом слое вычисленная активация основной модели $h_{pt}$ проецируется в размерность боковой сети и смешивается с её собственным состоянием через обучаемый скалярный гейт:

$$x \leftarrow \sigma(\alpha)\, x + \big(1 - \sigma(\alpha)\big)\, W_{down}\, h_{pt}$$

При этом выход оригинальной сети заменяется на выход этой доавбленной "лестинцы". Основная сеть (backbone) работает исключительно как экстрактор признаков, градиенты в него не заходят вообще, поэтому ее активации хранить не нужно — именно здесь и возникает экономия памяти, недоступная адаптерам.

<img src="img/ladder.png" width=150>

#### AttentionFusion

Метод **AttentionFusion** от [(Cao et al., 2022)](https://aclanthology.org/2022.findings-naacl.64/) из Amazon развивает эту идею.

В 2022 у Amazon есть NLU-система, обслуживающая голосового ассистента Alexa. Это не одна модель, это сотни навыков, каждый со своим интентом, и постоянно релизятся новые. Плюс языковые сложности, один и тот же навык нужно завести на немецком, испанском, французском и т.д. 

Полный файн-тюнинг под каждую комбинацию не подходит, ведь нужно хостить отдельную большую модель на каждую задачу. PEFT-методы решают проблему хранения, но не проблему связанности. Команда, добавляющая новый навык, по возможности не должна трогать общий энкодер.

Тогда обучаемый модуль решили разместить не внутри, не параллельно, а *после* замороженного энкодера. Модель агрегирует активации от всех слоев, в Attention стиле, как взвешенную сумму. Свой подход авторы харакетризуют как *late fusion*, в противовес адаптерам, софт-промптам и селективным методам, которые соответственно называют *early fusion*. Затем полученный вектор представлений прогоняется через небольшой MLP под специфическую задачу, под которую мы дообучаемся.

<img src="img/attention_fusion.png" width=450>

Слой агрегации, в терминологии авторов Fusion слой выглядит так. Мы заводим под каждую задачу свой обучаемый вектор $Q^t$. Для каждого токена входной последовательности мы вычисляем смесь слоев для данной задачи. Затем для каждого токена взвешиваем этой смесью V представления посчитанные оригинальной моделью и получаем выходное представление

$$\alpha_i^j(t) = \frac{\exp(Q_t V_i^j)}{\sum_k \exp(Q_t V_i^k)}, \qquad c_i(t) = \sum_j \alpha_i^j(t)\, V_i^j$$

Обратите внимание на зависимость внимания от токена, для разных токенов представления V смешивается по-разному.

 Результат отстает на пару процентнвх пунктов от файн-тюнинга, но авторы предполагают, что задачам вывода нужна иерархическая перестройка признаков, которую late fusion по построению дать не может. Заодно авторы прказали в своем исследовании, что распределение весов внимания слоев зависит от задачи, но почти не зависит от языка, поэтому обученный на английском модуль переносится на другие языки без дообучения.

#### Learn To Share

В методе **LeTS** (Learn To Share) от [(Fu, 2021)](https://proceedings.mlr.press/v139/fu21a.html) также комбинируем выход со всех слоев, но уже не в Attention стиле, а еще более гибко, через постройку архитектуры. В модели идут две параллельне ветки - "замороженная" предобученная и текущая обучаемая. 

Какая из двух веток пойдет дальше и какая из веток генерирует выход выбирает бинарный селектор. Выход вычисляется так: берем с каждого слоя предсталвение [CLS] токена, прогноняем его через линейную трансофрмацию Linear и последовательно прокручиваем все представления через рекуррентную Bi-LSTM сеть, начиная с первого.

<img src="img/lts1.png" width=400>

Кроме того, обучаемую ветку еще и разреживаем методом DeltaPruning. 

Выбор селектора реализуется через подход хорошо известный в домене NAS (neural architecture search), и называемый DARTS. Идея DARTS заключается в том, что мы сначала собираем модель со всеми опциями (так называемая "супермодель") и добавляем дискретный селектор между опциями. Чтобы не прибегать к полному перебору, каждый дискретный переключатель заменяется непрерывным дифференцируемым приближением. В методе LeTS в проли такого приблжиения выступает взвешивание по Gumbalt: сигналы всех веток объединяются как взвешенная сумма. Далее уже запускается стандартная градиентная оптимизация.

NAS это крайне затратный метод, потому модель не снискала особенную популярность.

## Адаптеры

### Введение
Идея данной группы методов в следующем - давайте вставлять компактные обучаемые модули внутрь блоков трансформера, которые будут корректировать сигнал с помощью выученной "поправки" к предобученной модели. Перед дообучением основные веса модели замораживаются, а обучаются только эти поправки.

Ключевое требование к такому модулю — он должен стартовать как тождественное преобразование, иначе вставка сломает предобученные представления ещё до первого шага оптимизации. Поэтому почти все методы группы инициализируют модуль так, чтобы на старте он возвращал вход без изменений, и дальше постепенно отклоняются от тождества.

К сильным сторонам данной группы можно отнести модульность в самом буквальном виде: адаптер — это отдельный физический объект, его можно обучить, сохранить, скачать и подключить, не трогая оригинальную модель. Именно на адаптерах выросла инфраструктура вроде AdapterHub, репозиторий предгбученных адаптеров и именно здесь появились содержательные схемы композиции нескольких навыков. Кроме того, обучение адаптеров устойчиво при малых данных: замороженная база работает как сильная регуляризация, поэтому на маленьких датасетах адаптеры нередко обходят полный файн-тюнинг.

Основная слабая сторона это вычислительный оверхед на этапе инференса. Если адаптер встроен в вычислительный граф последовательно, то модель фактически становится в два раза глубже, и каждый блок обрастает дополнительной задержкой (на запуск вычисления на видеокарте). Вмерджить рассчитанную прибавку в оригинальную модель не получится, так как между двумя проекциями стоит нелинейность.

### Обзор методов
Одна из первых работ принадлежит [(Houlsby, 2019)](https://arxiv.org/abs/1902.00751), которые описали архитектуру адаптера (**Adapter**) и которая дала старт целой линейке методов. В данной работе адаптер это двухслойная MLP сеть, собранная по принципу "бутылочного горлышка" (размерность посередине меньше входной и выходой):

$$f(h W_{down}) W_{up}$$

В модели Houlsby адаптер вставляется в каждый Трансформерный слой в двух местах: на выходе Attention-модуля (но до слоя нормализации) и на выходе FFN модуля (но до слоя нормализации).

<img1 src="img/adapters_houlsby.png" width=600>

$$h \leftarrow f(\text{Attn}(x) W_{down}) W_{up}$$

Обучается всего 3% параметров и при этом дает только -0.4% по качеству (сравнивали BERT модель на GLUE)

[(He et al, 2021)](https://arxiv.org/abs/2110.04366) решили использовать точно такой же адаптер, но подключают его параллельно, а не последовательно. Модель получила соотвествующее название **Parallel Adapters**. На примере слоя внимания:

<img1 src="img/adapters_parallel.png" width=600>

$$h \leftarrow \underbrace{\text{Attn}(x)}_{\text{слой внимания}} + s \cdot \underbrace{f(x W_{down}) W_{up}}_{\text{адаптер}}$$

Данное изменение во-первых, убрало "конвеерность", ветви теперь независимы и могут считаться параллельно. Во-вторых, на тесты показывают, что параллельная схема стабильно выигрывает, особенно это видно для FFN слоя.

Далее методы, которык делают несколько модулей вместо одного.

**AdapterFusion** [(Pfeiffer et al, 2021)](https://arxiv.org/pdf/2005.00247) предложили механизм эффективного комбинирования нескольких адаптеров. Кроме того, в этой же работе они показали, что для достижения того же уровня достаточно одного адаптера, размещенного после FFN модуля. Кроме того, LayerNorm слой тоже заморозили<br><img src="img/adapters_pfeiffer.png" width=200><br><br>

**AdaMix** [(Wang et al, 2021)](https://arxiv.org/abs/2205.12410) размножили адаптер на несколько параллельных версий. При обуении сигнал посылается случайно в одну из них. На инференс выход всех адаптеров усреднияется смесь нескольких адаптеров в духе mixture-of-experts; на инференсе усредняются

<img1 src="img/adamix1.png" width=600>

Следующие методы основаны на удешевлении вычисления адаптера.

[(He et al., 2022)](https://arxiv.org/abs/2210.04284) в **Sparse Adapter** частично заимствует идею из селективных методов: к модели добавляют классический адаптер, но прунят его на препроцессинге (по результатам прогона нескольких батчей), по критериям вроде SNIP, magnitude или GraSP, на тестах лучше всего сработал [SNIP](https://arxiv.org/abs/1810.02340)).

<img1 src="img/adapters_sparse.png" width=500>

Зачем нужно дополнтельное разреживание, если адаптер и так изначально небольшой? Во-первых, это регуляризация сигнала. Актуально, когда мало данных для дообучения. Во-вторых, принцип, который авторы назвали <i>Large-Sparse</i>: при одном и том же кол-ве параметров выгоднее иметь большую разреженную модель, чем мальенкую плотную.

[(Karimi, 2021)]( https://arxiv.org/abs/2106.04647) попытались "выжать" маскимум экономии в своей версии адаптера **Compacter**. Первое, что они сделали - заменили матрицы весов $W_{down}$ и $W_{up}$ на кронекровское произведение $A \bigotimes B$, а точнее как сумму нескольких таких произведений $\sum A_i \bigotimes B_j$. Напомним, что есть кронекеровское произведение. Пусть $A$ - малая матрица, $B$ - матрица побольшеб тогда

<img src="img/krona.png" width=250>

Уже на этом этапе кол-во параметров довольно существенно скоращается. Но далее они сделали все матрицы $A_i$ общими в рамках модели. И кроме того, все матрицы B приблизили одноранговым произведением двух векторов $B=p\times q$

<img1 src="img/adapters_compacter1.png" width=500>

Если возьмем конкретный пример, то замена на кронекеровское произведение сокращает ко-во праметров с $64 × 768 = 49152$ до $(4 × 4) \bigotimes (16 × 192) = 3088$. Пусть мы раскладываем в сумму из 4 пар, тогда суммарно это будет 12355 параметров. Наконец, $B$ заменяем одноранговым произведением и получаем $16 + 4 x (10+192) = 832$ параметра вместо 49355. Сжатие порядка 59 раз.

[(Lui et al, 2022)](https://arxiv.org/abs/2205.05638) подошли к вопросу экономии еще более радикально. В своей модели **(IA)³** (Infused Adapter by Inhibiting and Amplifying Inner Activations) авторы встраивают адаптер не сразу после а прямо внутрь механизма внимания. И адаптер используют не классичекий, а просто масштабирующий коэффицент (Scaler), усиливающий или ослабляющий активации. Адаптер добавляется в трех местах: на выходе K проектора, на выходе V проектора, а также на выходе слоя FFN.

<img1 src="img/adapters_ia1.png" width=300>

На обучении коээффициент масшиабирования подбирается, а после окончания обучения он вмердживается в матрицу весов (так как композиция линейных слоев = линейный слой)

<img1 src="img/adapters_ia2.png" width=300>


## Soft prompts методы
Идея: давайте новое "знание" не прибавлять к весам модели, как это делают адаптеры, а *конкатенировать* его с весами модели. И находясь уже в векторных описаниях это знание будет замешиваться с входным сигналом через механизм внимания (Self-Attention), как будто это обычные параметры. Конкатенировать будем со входом, но не с самим текстовым промптом, а с первым слоем - таблицей сырых эмбедингов

Визуально процесс похож на добавление в промпт новых воодных, поэтому назвали prompting. А поскольку конкатенируются не сырые токены, а их эмбединги, то "soft prompting". В литературе присоединяемые эмбединги называют иногда псевдотокенами или виртуальными токенами

Несколько популярных подходов:

**Prompt-tuning**  [(Lester et al, 2021)](https://arxiv.org/abs/2104.08691) Простейший вариант. Заводится обучаемая матрица `P` размером `[k × d]` — это `k` «виртуальных токенов» (обычно 10–100) размерности модели `d`. На прямом проходе они *приписываются спереди* к эмбеддингам реального входа, и объединённая последовательность длины `k + n` идёт через трансформер как обычно; в attention-слое реальные токены «видят» префикс (как будто обычные токены) и подстраиваются под него. Обучается только `P`, добавление происходит один раз — на самом входе<br><img src="img/prompt.png" width=150>

**Prefix-tuning**  [(Li et al, 2021)](https://arxiv.org/abs/2101.00190) То же самое, только обучаемые векторы добавляются *на каждом слое*, причём прямо в механизм внимания: к реальным key и value приклеиваются обучаемые префиксные K/V (свой набор на слой)<br><img src="img/prefix.png" width=200>

**P-tuning**  [(Liu ert al, 2021)](https://arxiv.org/abs/2103.10385)<Br>Во время обучения модели эмбединги псевдотокенов перед конкатенацией с эмбедингами реальных токенов прогнояются через небольшую MLP сеть (таким образом взаимодейству/ют друг с другом и становятся контекстно-зависимыми). После обучения в итоговый словарь вставляют уже "провернутую" версию эмбедингов вместо оригинальных. Эта манипуляция делается для стабилизации обучения, получается лучший performance на конечный задачах (хз почему, но вероятно, благодаря MLP подбирается более оптимальная "геометрия" сигнала)<br><img src="img/ptuning.png" width=300>

__WARP__ [(Hambardzumyan et al. 2021)](https://arxiv.org/abs/2101.00121)<br>то же самое, что Prompt Tuning, но добавляется небольшое кол-во токенов (1-5) + добавляется еще HEAD.

__SPoT__ [(Vu et al, 2022)](https://arxiv.org/pdf/2110.07904)<br>перенос выученных soft-prompt'ов с задачи на задачу как инициализация

## Методы репараметризации
Идея: мы не добавляем новые слои в сеть (как adapters) и не приписываем токены ко входу (как soft prompts), а перепараметризуем поправку к уже существующим весовым матрицам через произведение меньших матриц. Поправка `ΔW` к исходной матрице `W` представляется в компактной факторизованной форме, что резко снижает число обучаемых параметров. Базовые веса при этом заморожены.

### Intrinsic-SAID
В 2021 году [(Aghajanyan et al)](https://arxiv.org/abs/2012.13255) провели большое исследование, в котором показали, что у предобученных моделей очень низкая «внутренняя размерность» (intrinsic dimension). Это значит, что должна существовать низкоразмерная репараметризация, столь же эффективная для дообучения, как и полное пространство параметров. Предложили модель SAID или Structure-Aware Intrinsic Dimension.

В рамках модели апдейт $\Delta W$ моделируется как "upscale" проекция из компактного вектора $θ^{d}$. Проекция $P: \mathbb{R}^d \rightarrow \mathbb{R}^D$ - случайная проекция, генерируемая один раз, реализованная через FastFood преобразование (математический трюк для быстрой генерации случайных матриц). В базовом варианте (DID: direct ID) одна проекция шарится между всеми слоями. В Structure-Aware варианте она для каждого слоя своя

Fastfood трансформация выглядит вот так: $F(w_r) = \frac{1}{\sigma \sqrt{n}}BH\Pi GS$, где<br>
B – diagonal matrix with random ±1 entries<br>
H – Walsh–Hadamard matrix<br>
Π – random permutation matrix<br>
G – diagonal Gaussian random variables<br>
S – scaling factors for variance control<br>
σ – normalization constant

### LoRA
Метод LoRA или Low-Rank Adaptation был предложен в 2021 году авторами из Microsoft [(Hu et al, 2021)](https://arxiv.org/abs/2106.09685) и стал де-факто стандартом дообучения LLM

Идея метода в том, что апдейт $\Delta W$ низкоранговый, а значит его можно апроксимировать произведением двух "узких" матриц A и B и обучать только их:$$h = W_{\theta}x + \Delta Wx = W_{\theta}x + (B \cdot A)x$$, где матрица $A \in \mathbb{R}^{(r×k)}$ - «up-projection», сжимает вход в ранг `r`, матрица $B \in \mathbb{R}^{(d×r)}$ — «up-projection», разворачивает обратно<br><img src="img/lora1.jpg" width=350>

Есть очевидное сходство с адаптерами, а точнее с паралельным его вариантом (Parallel adapter). Единственное отличие: LoRA это линейное преобразование, в то время как параллельный адаптер имеет актвиацию между слоями<br><img src="img/lora_adapter.png" width=350>

Кроме того поправка иногда масштабируется коэффициентом $α/r$: $$h = W_{\theta}x + (α/r) \cdot B \cdot A \cdot x$$
Исследование авторов показало, что LoRA разложение выгоднее применять к Attention матрицам $W_q$ и $W_v$<Br><br>
Матрица $A$ инициализируется Гауссовым распредедением $A \propto N(0, \sigma^2)$, а матрица $B$ — нулями. То есть `BA = 0`, и модель начинает с нуля накапливать изменение

После обучения можно слить с матрицами, либо хранить `{A, B}` отдельно как подключаемый адаптер<br><br>
Для GPT-3-175B LoRA снижает число обучаемых параметров в 10000 раз; потребление GPU-памяти в 3 раза; при том же качестве обучения. Ближайший конкурент - Houlsby адаптеры. На некоторых задачах качество даже выше fine-tuning.
<img1 src="img/lora2.png" width=550>

### KronA 
[(Edalati et al, 2022))](https://arxiv.org/pdf/2212.10650) из Huawei предложили модификацию принципа LoRA которую назвали KronA или Kronecker Adapter. В рамках этой модифкации вместо матричного произведения используется произведение Кронекера. Напомним, что есть произведение Кронекера

<img src="img/krona.png" width=250>

Проблема с LoRA в том, что ранг поправки жёстко ограничен сверху значением $r$. У произведения Кронекера есть хорошее свойство, что оно сохраняет ранг перемножаемых матриц: $rank(A \bigotimes B) = rank(A) \cdot rank(B)$.

Линейный слой с KronA: $h = W_{\theta}x + s·(A_k \bigotimes B_k) \cdot x$, где $s$ - масштабирующий коэффициент как в LoRA.

Адаптируется как Attention, так и FFN. В FFN добавляется residual connection. После обучения сливаем $\Delta W$ с $W_{\theta}$, либо если `KronAᴮ_res`, то оставить как параллельную ветвь с residual connection.

<img1 src="img/krona_lora1.png" width=350>

<img1 src="img/krona_lora2.png" width=350>

### QLoRA
Еще одно важное развитие семейтсва это QLoRA. Идея: чтобы экономить память ещё сильнее, базовую модель **квантуют** — хранят её веса в очень низкой точности (4 бита вместо 16). Базовая модель при этом заморожена, а LoRA-адаптеры обучаются поверх неё в более высокой точности.
<br>Результат: дообучение очень крупных моделей становится возможным на одной потребительской видеокарте. QLoRA фактически демократизировала fine-tuning — то, что раньше требовало кластера, стало доступно энтузиастам

Количество обучаемых параметров падает с `d²` до `2 · d · r`. При `d = 4096` и `r = 8` это уменьшение примерно в 256 раз.
<br>Переключаемость - можно обучить много разных LoRA-«адаптеров» под разные задачи (каждый весит мегабайты) и подгружать их к одной и той же базовой модели по необходимости. Это снимает проблему хранения десятков полных копий.

## Гибридные подходы
Не самостоятельный принцип, а методы на пересечении семейств. Два типичных сценария: ручное объединение механизмов в единый фреймворк (например, prefix-tuning + адаптеры) и обучаемое смешивание, когда метод сам через gating подбирает, какие механизмы и в какой пропорции включить. Отдельная ветвь — систематический поиск по «дизайн-пространству» PEFT.

**MAM Adapter** [He et al, 2022](https://arxiv.org/abs/2110.04366)<br>разные части трансформера лучше адаптируются разными механизмами — внимание префиксами, FFN адаптером большой ёмкости

**UniPELT** [(Mao et al, 2022)](https://arxiv.org/abs/2110.07577)<br>Метод Unipelt = A Unified Framework for Parameter-Efficient Language Model Tuning. Это попытка создать универсальное решение: в каждый блок трансформера встроены три PEFT-подмодуля сразу: LoRA + prefix + adapters, и над каждым стоит обучаемый «вентиль» (gate), определяющий его вклад<br><img src="img/unipelt.png" width=300><br><br>

**IPT** [(Qin, 2022)](https://arxiv.org/abs/2110.07867)<br>Модель IPT=Intrinsic Prompt Tuning. Берем 100 задач и обучаем soft prompts для каждой. Учим на 100 точках автоэнкодер. Замораживае м получившийся декодер. сжимаем эти софт промпты в вектор

**S4** [Chen, 2023](https://arxiv.org/abs/2301.01821)<br>результат систематического поиска по «дизайн-пространству» PEFT

**Sparse LoRA**<br>LoRA с разреженностью (selective + reparametrization)


## Данные

В зависимости от того, чего мы хотим добиться:
- Continued pre-training (доменная адаптация)<br>Просто большой объём «сырого» текста из целевого домена (например, корпус юридических документов). Цель — пропитать модель доменным языком и фактами. Разметка не нужна<br><br>
- Supervised fine-tuning, SFT / instruction tuning<br>Пары «инструкция (вход) → желаемый ответ (выход)». Это основной формат для обучения модели следовать указаниям. Именно здесь формируется поведение «ассистента»<br><br>
- Данные предпочтений (preference data)<br>Тройки вида «запрос + лучший ответ + худший ответ». Используются на следующей стадии (выравнивание), которая выходит за рамки этой темы.

Для SFT часто также добавляют системный промпт и структуру диалога (роли «пользователь»/«ассистент»), чтобы модель училась работать в чат-формате

Отдельно стоит подчеркнуть значимость instruction tuning. Ранний fine-tuning настраивал модель под одну узкую задачу. Прорыв состоял в том, чтобы дообучить модель на смеси *множества разнообразных задач, сформулированных как инструкции на естественном языке*.

Результат оказался неожиданным: модель, обученная следовать инструкциям на наборе известных задач, начинает обобщать это умение и разумно реагировать на новые, не виденные ранее инструкции. Так «языковая модель, предсказывающая токены» превращается в «ассистента, выполняющего запросы». Так языковые модели стали привычным для нас чат-ботами.

Способы сбора данных для дообучения:

- Ручная разметка людьми<br>Эксперты или асессоры пишут эталонные ответы на запросы. Самый дорогой, но и самый качественный способ; хорош для задач, требующих экспертизы или аккуратности. Часто используется для сравнительно небольших, но «золотых» наборов.

- Сбор из существующих источников<br>Переиспользование уже имеющихся данных: логи поддержки, базы вопросов-ответов, документация, существующие датасеты. Дёшево, но требует очистки и приведения к нужному формату

- Синтетическая генерация (LLM-generated data)<br>Современный и очень популярный подход: использовать более сильную модель, чтобы сгенерировать обучающие данные для целевой. Типичный приём — дать мощной модели несколько примеров-«затравок» и попросить нагенерировать тысячи разнообразных пар «инструкция → ответ» (подход в духе Self-Instruct). Это дёшево и масштабируемо

- Важные оговорки по синтетике: нужно следить за разнообразием (модель склонна повторяться) и за накоплением ошибок (если генератор ошибается, его ошибки попадут в выборку). Часто синтетику комбинируют с человеческой фильтрацией.

- Distillation (дистилляция)<br>Частный случай синтетики: обучаем меньшую модель на ответах большей «учительской» модели, перенося её поведение. Многие открытые инструктивные модели обучены именно так.

Ключевой эмпирический вывод последних лет: для instruction tuning качество и разнообразие данных важнее их объёма. Несколько тысяч тщательно отобранных, разнообразных и чистых примеров часто дают лучший результат, чем сотни тысяч шумных. Это породило отдельное направление работы — отбор и фильтрацию данных (data curation): дедупликация, отсев низкокачественных и токсичных примеров, балансировка по типам задач, контроль длины и сложности.

При сборе выборки также важно следить за:
- покрытием - представлены ли все типы запросов, которые встретятся в проде;
- балансом - не доминирует ли один тип задач;
- утечками - не попали ли в обучение тестовые примеры (это завысит метрики);
- форматной согласованностью — единый стиль и структура ответов
